In [95]:
# This notebook combines master datasets from multiple countries into a unified dataset for analysis
import numpy as np
import pandas as pd

df_in_master = pd.read_parquet("INDIA/DATA/4.IN_MASTER_DATASET.parquet", engine="pyarrow")
df_mm_master = pd.read_parquet("MYANMAR/DATA/4.MM_MASTER_DATASET.parquet", engine="pyarrow")
df_pk_master = pd.read_parquet("PAKISTAN/DATA/4.PK_MASTER_DATASET.parquet", engine="pyarrow")


In [96]:
all_cols_master = sorted(
    set(df_mm_master.columns)
    | set(df_pk_master.columns)
    | set(df_in_master.columns)
)

df_all_master = pd.concat(
    [
        df_mm_master.reindex(columns=all_cols_master),
        df_pk_master.reindex(columns=all_cols_master),
        df_in_master.reindex(columns=all_cols_master)
    ],
    axis=0,
    ignore_index=True
)

In [97]:
#normalize everything to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex", 
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
    "hv247HasBankAccount",
]
for col in cols_to_numeric:
    df_all_master[col] = pd.to_numeric(
        df_all_master[col],
        errors="coerce"  
    )

In [98]:
print("\nDtypes:")
print(df_all_master.dtypes)


Dtypes:
745aHouseOwnership                  float64
Depr_CleanFuel                        int32
Depr_Elec                             int32
Depr_Kitchen                          int32
Depr_Phone                            int32
Depr_Refrigerator                     int32
Depr_TV                               int32
EnergyPoor                             int8
HouseholdClusterElevation           float64
MEPI_score                          float64
hhidCaseIdentification       string[python]
hv000CountryCode                     object
hv001ClusterNumber                    int32
hv002HouseholdNumber                  int32
hv005SimplilingWeight                 int32
hv009FamilySize                        int8
hv014NoOfChildren                      int8
hv024RegionDivision                    int8
hv025TypeOfResidence                   int8
hv106_01EducationOfHead             float64
hv115_01MaritalStatus               float64
hv206HasElectricity                 float64
hv208HasTelevision     

In [99]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")
    df[[
        "EnergyPoor",
    ]] = df[[
        "EnergyPoor",
    ]].astype("int8")

    return df

In [100]:
print("\nDtypes:")
#print(df_all_train_8.dtypes)

print("\nMissing values:")
print(df_all_master.isna().sum().sort_values(ascending=False).head())


Dtypes:

Missing values:
hv242SeparateKitchenYesNo    140529
HouseholdClusterElevation     14446
hv115_01MaritalStatus           119
hv106_01EducationOfHead           3
hv247HasBankAccount               2
dtype: int64


In [101]:
df_all_master_main = restore_ml_dtypes(df_all_master)

In [102]:
print("\nDtypes:")
print(df_all_master_main.dtypes)


Dtypes:
745aHouseOwnership                 category
Depr_CleanFuel                        int32
Depr_Elec                             int32
Depr_Kitchen                          int32
Depr_Phone                            int32
Depr_Refrigerator                     int32
Depr_TV                               int32
EnergyPoor                             int8
HouseholdClusterElevation           float64
MEPI_score                          float64
hhidCaseIdentification       string[python]
hv000CountryCode                     object
hv001ClusterNumber                    int32
hv002HouseholdNumber                  int32
hv005SimplilingWeight                 int32
hv009FamilySize                        int8
hv014NoOfChildren                      int8
hv024RegionDivision                category
hv025TypeOfResidence               category
hv106_01EducationOfHead            category
hv115_01MaritalStatus              category
hv206HasElectricity                 float64
hv208HasTelevision     

In [103]:
print("Value counts:")
print(df_all_master_main["hv247HasBankAccount"].value_counts(dropna=False))

print("\nDtype:")
print(df_all_master_main["hv247HasBankAccount"].dtype)

Value counts:
hv247HasBankAccount
 1.0    615231
 0.0     47106
 8.0       148
NaN          2
Name: count, dtype: int64

Dtype:
category


In [104]:
df_all_master_main["hv247HasBankAccount"] = df_all_master_main["hv247HasBankAccount"].astype("category")
df_all_master_main["hv247HasBankAccount"] = (
    df_all_master_main["hv247HasBankAccount"]
    .cat.add_categories("Missing")
    .fillna("Missing")
)

In [105]:
print("Value counts:")
print(df_all_master_main["hv106_01EducationOfHead"].value_counts(dropna=False))

print("\nDtype:")
print(df_all_master_main["hv106_01EducationOfHead"].dtype)

Value counts:
hv106_01EducationOfHead
 2.0    276121
 0.0    197635
 1.0    124839
 3.0     63303
 8.0       586
NaN          3
Name: count, dtype: int64

Dtype:
category


In [106]:
df_all_master_main["hv106_01EducationOfHead"] = df_all_master_main["hv106_01EducationOfHead"].astype("category")
df_all_master_main["hv106_01EducationOfHead"] = (
    df_all_master_main["hv106_01EducationOfHead"]
    .cat.add_categories("Missing")
    .fillna("Missing")
)

In [107]:
print("Value counts:")
print(df_all_master_main["hv115_01MaritalStatus"].value_counts(dropna=False))

print("\nDtype:")
print(df_all_master_main["hv115_01MaritalStatus"].dtype)

Value counts:
hv115_01MaritalStatus
 1.0    555213
 3.0     86320
 0.0     13303
 5.0      4747
 4.0      2753
NaN        119
 8.0        32
Name: count, dtype: int64

Dtype:
category


In [108]:
df_all_master_main["hv115_01MaritalStatus"] = df_all_master_main["hv115_01MaritalStatus"].astype("category")
df_all_master_main["hv115_01MaritalStatus"] = (
    df_all_master_main["hv115_01MaritalStatus"]
    .cat.add_categories("Missing")
    .fillna("Missing")
)

In [109]:
print(df_all_master_main.isna().sum().sort_values(ascending=False).head())

hv242SeparateKitchenYesNo    140529
HouseholdClusterElevation     14446
745aHouseOwnership                0
hv221HasLandLine                  0
hv206HasElectricity               0
dtype: int64


In [110]:
cols_to_string = [
    "hv106_01EducationOfHead","hv115_01MaritalStatus","hv247HasBankAccount",
]
df_all_master_main[cols_to_string] = df_all_master_main[cols_to_string].astype("string")

In [111]:
# 4. Save
df_all_master_main.to_parquet(
    r"COMBINE\DATA\1.CB_MASTER_DATASET.parquet",
    engine="pyarrow",
    compression="snappy", index=False
)

print(f"✅ File saved.")

✅ File saved.
